# 🎯 Ćwiczenie 9: MLlib — Predykcja kliknięć w reklamy (CTR)

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 9
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐⭐⭐⭐ (Zaawansowany)

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:
1. Przygotować zmienne kategoryczne pod ML za pomocą `StringIndexer` i `OneHotEncoder`
2. Zbudować pełny `Pipeline` MLlib łączący preprocessing i model
3. Wytrenować i porównać `LogisticRegression` oraz `RandomForestClassifier`
4. Ewaluować modele przy pomocy `BinaryClassificationEvaluator` (AUC) i `MulticlassClassificationEvaluator` (accuracy, precision, recall)
5. Zapisać i wczytać wytrenowany `PipelineModel`

## 📋 Wymagania wstępne
- Znajomość PySpark DataFrame (Wykłady 4-6)
- Znajomość scikit-learn: Pipeline, klasyfikacja, ewaluacja (Wykład 8)

---
> 💡 **Tip:** Uruchamiaj komórki po kolei (`Shift+Enter`). Jeśli utkniesz, przeczytaj komórki z podpowiedziami (💡 Hint) lub zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — uruchom tę komórkę jako pierwszą!

!pip install -q pyspark

import random
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

random.seed(42)
np.random.seed(42)

spark = (
    SparkSession.builder
    .appName("AdBoost-CTR-Prediction")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("✅ Setup complete! SparkSession gotowa 🚀")
print(f"Wersja Sparka: {spark.version}")

## 📖 Kontekst: AdBoost 📱🖱️

Pracujesz jako data scientist w agencji reklamowej **AdBoost**. Zespół sprzedaży chce wiedzieć, które reklamy mają największe szanse na kliknięcie (CTR — *Click-Through Rate*) — zanim jeszcze zostaną wyświetlone. Masz ~300 000 wyświetleń reklam z ostatniego miesiąca. To zdecydowanie za dużo, by wygodnie iterować w scikit-learn na laptopie — czas na **Spark MLlib**!

### Kluczowe koncepty MLlib

| Koncept | Opis | Przykład |
|---------|------|----------|
| `StringIndexer` | koduje string → indeks liczbowy | `StringIndexer(inputCol="device_type", outputCol="device_type_idx")` |
| `OneHotEncoder` | koduje indeks → wektor rzadki (bez fałszywej kolejności) | `OneHotEncoder(inputCol="device_type_idx", outputCol="device_type_ohe")` |
| `VectorAssembler` | łączy kolumny w jedną kolumnę wektorową `features` | `VectorAssembler(inputCols=[...], outputCol="features")` |
| `Pipeline` | łańcuch Transformerów/Estimatorów, `fit()` → `PipelineModel` | `Pipeline(stages=[...])` |
| `BinaryClassificationEvaluator` | mierzy AUC dla klasyfikacji binarnej | `evaluator.evaluate(predictions)` |

In [ ]:
# @title 📊 Generowanie zbioru danych CTR (~300 000 wyświetleń)

N = 300_000

ad_categories = ["moda", "elektronika", "podroze", "jedzenie", "sport", "finanse", "uroda", "gry"]
times_of_day = ["rano", "poludnie", "popoludnie", "wieczor", "noc"]
device_types = ["mobile", "desktop", "tablet"]
ad_positions = ["top", "middle", "bottom"]
age_groups = ["18-24", "25-34", "35-44", "45-54", "55+"]
days_of_week = ["poniedzialek", "wtorek", "sroda", "czwartek", "piatek", "sobota", "niedziela"]

ad_category = np.random.choice(ad_categories, N)
time_of_day = np.random.choice(times_of_day, N, p=[0.15, 0.20, 0.25, 0.25, 0.15])
device_type = np.random.choice(device_types, N, p=[0.55, 0.35, 0.10])
ad_position = np.random.choice(ad_positions, N, p=[0.30, 0.40, 0.30])
age_group = np.random.choice(age_groups, N)
day_of_week = np.random.choice(days_of_week, N)

# Realistyczne korelacje: pozycja "top" i urządzenie "mobile" zwiększają CTR
base_prob = 0.05
position_bonus = np.where(ad_position == "top", 0.08, np.where(ad_position == "middle", 0.02, 0.0))
device_bonus = np.where(device_type == "mobile", 0.05, np.where(device_type == "tablet", 0.02, 0.0))
time_bonus = np.where(np.isin(time_of_day, ["popoludnie", "wieczor"]), 0.03, 0.0)
weekend_bonus = np.where(np.isin(day_of_week, ["sobota", "niedziela"]), 0.02, 0.0)

click_prob = np.clip(base_prob + position_bonus + device_bonus + time_bonus + weekend_bonus, 0.0, 0.95)
clicked = (np.random.rand(N) < click_prob).astype(int)

pdf = pd.DataFrame({
    "impression_id": np.arange(1, N + 1),
    "ad_category": ad_category,
    "time_of_day": time_of_day,
    "device_type": device_type,
    "ad_position": ad_position,
    "age_group": age_group,
    "day_of_week": day_of_week,
    "clicked": clicked,
})

df = spark.createDataFrame(pdf)
df.printSchema()
print(f"📊 Liczba wierszy: {df.count():,}")
df.groupBy("clicked").count().show()

## 🎓 Guided Example

Zbudujmy razem pełny `Pipeline` MLlib: `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `LogisticRegression`, a następnie oceńmy go metryką AUC.

**Uwaga:** Ten przykład używa własnego, lokalnego podziału danych (`guided_train` / `guided_test`) tylko do demonstracji. W ćwiczeniach stworzysz **własny** oficjalny podział (`train_df` / `test_df`), którego użyjesz we wszystkich kolejnych zadaniach.

In [ ]:
# @title 🎓 Guided Example — uruchom i przestudiuj

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

categorical_cols = ["ad_category", "time_of_day", "device_type", "ad_position", "age_group", "day_of_week"]
indexed_cols = [f"{c}_idx" for c in categorical_cols]
ohe_cols = [f"{c}_ohe" for c in categorical_cols]

# Krok 1: StringIndexer dla każdej kolumny kategorycznej
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]

# Krok 2: OneHotEncoder dla każdego indeksu (unikamy fałszywej kolejności kategorii)
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]

# Krok 3: VectorAssembler łączy wszystkie zakodowane kolumny w jedną kolumnę "features"
assembler = VectorAssembler(inputCols=ohe_cols, outputCol="features")

# Krok 4: model klasyfikacji
lr = LogisticRegression(featuresCol="features", labelCol="clicked", maxIter=20)

pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

guided_train, guided_test = df.randomSplit([0.7, 0.3], seed=42)

pipeline_model = pipeline.fit(guided_train)          # 👈 trenuje CAŁY łańcuch naraz
guided_predictions = pipeline_model.transform(guided_test)

evaluator = BinaryClassificationEvaluator(
    labelCol="clicked", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
guided_auc = evaluator.evaluate(guided_predictions)

print(f"✅ Guided example — LogisticRegression AUC: {guided_auc:.4f}")
guided_predictions.select("ad_position", "device_type", "clicked", "prediction", "probability").show(5, truncate=False)

---

## ✏️ Ćwiczenie 1: Oficjalny podział train/test

**Trudność:** ⭐

Podziel `df` na `train_df` (70%) i `test_df` (30%), używając ziarna losowości `seed=42`. Te dwie zmienne będą używane we **wszystkich** kolejnych ćwiczeniach.

**Oczekiwany wynik:** `train_df` ma ok. 70% wierszy, `test_df` ok. 30% wierszy, a ich suma równa się `df.count()`.

> 💡 **Hint:** Użyj `df.randomSplit([...], seed=...)`.

In [ ]:
# ✏️ Ćwiczenie 1: Podział train/test

# TODO: podziel df na train_df (70%) i test_df (30%), seed=42
train_df = ...  # Zastąp ... swoim kodem
test_df = ...  # Zastąp ... swoim kodem

print(f"train_df: {train_df.count():,} wierszy")
print(f"test_df: {test_df.count():,} wierszy")

In [ ]:
# @title ✅ Test — Ćwiczenie 1

try:
    train_count = train_df.count()
    test_count = test_df.count()
    total = df.count()

    assert train_count is not ..., "❌ train_df nie zostało uzupełnione"
    assert abs(train_count / total - 0.7) < 0.02, f"❌ train_df powinno mieć ok. 70% wierszy, ma {train_count / total:.2%}"
    assert abs(test_count / total - 0.3) < 0.02, f"❌ test_df powinno mieć ok. 30% wierszy, ma {test_count / total:.2%}"
    assert train_count + test_count == total, "❌ suma train_df + test_df powinna równać się df.count()"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 2: Pipeline z RandomForestClassifier

**Trudność:** ⭐⭐

Zbuduj `Pipeline` analogiczny do guided example, ale zamień `LogisticRegression` na `RandomForestClassifier` (`numTrees=100`, `maxDepth=8`). Wytrenuj go na `train_df` i wygeneruj predykcje na `test_df`.

**Oczekiwany wynik:** `rf_predictions` to DataFrame z kolumną `prediction`, o tej samej liczbie wierszy co `test_df`.

> 💡 **Hint:** Możesz ponownie użyć `categorical_cols` i `ohe_cols` z guided example — zmienia się tylko ostatni stage Pipeline'u.

In [ ]:
# ✏️ Ćwiczenie 2: RandomForestClassifier

from pyspark.ml.classification import RandomForestClassifier

rf_indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
rf_encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]
rf_assembler = VectorAssembler(inputCols=ohe_cols, outputCol="features")

# TODO: stwórz RandomForestClassifier(featuresCol="features", labelCol="clicked", numTrees=100, maxDepth=8)
rf_classifier = ...  # Zastąp ... swoim kodem

rf_pipeline = Pipeline(stages=rf_indexers + rf_encoders + [rf_assembler, rf_classifier])

# TODO: wytrenuj pipeline na train_df
rf_model = ...  # Zastąp ... swoim kodem

# TODO: wygeneruj predykcje na test_df
rf_predictions = ...  # Zastąp ... swoim kodem

print(f"Liczba wierszy w rf_predictions: {rf_predictions.count():,}")

In [ ]:
# @title ✅ Test — Ćwiczenie 2

try:
    assert rf_classifier is not ..., "❌ rf_classifier nie zostało uzupełnione"
    assert "prediction" in rf_predictions.columns, "❌ Brak kolumny 'prediction' w rf_predictions"
    assert rf_predictions.count() == test_df.count(), "❌ rf_predictions powinno mieć tyle samo wierszy co test_df"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 3: Porównanie AUC — LogisticRegression vs RandomForest

**Trudność:** ⭐⭐

Oblicz AUC dla predykcji `LogisticRegression` (przelicz `pipeline_model` z guided example na oficjalnym `test_df`) oraz dla `rf_predictions` z Ćwiczenia 2. Zapisz wyniki jako `lr_auc` i `rf_auc`.

> 💡 **Hint:** Ponownie użyj `pipeline_model.transform(...)` oraz istniejącego obiektu `evaluator`.

In [ ]:
# ✏️ Ćwiczenie 3: Porównanie AUC

# TODO: wygeneruj predykcje LogisticRegression na oficjalnym test_df
lr_predictions_test = ...  # Zastąp ... swoim kodem

# TODO: oblicz AUC dla obu modeli za pomocą evaluator.evaluate(...)
lr_auc = ...  # Zastąp ... swoim kodem
rf_auc = ...  # Zastąp ... swoim kodem

print(f"LogisticRegression AUC: {lr_auc:.4f}")
print(f"RandomForest AUC:      {rf_auc:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 3

try:
    assert lr_auc is not ..., "❌ lr_auc nie zostało uzupełnione"
    assert rf_auc is not ..., "❌ rf_auc nie zostało uzupełnione"
    assert 0.0 <= lr_auc <= 1.0, "❌ lr_auc powinno być w zakresie [0, 1]"
    assert 0.0 <= rf_auc <= 1.0, "❌ rf_auc powinno być w zakresie [0, 1]"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 4: Ważność cech (feature importances)

**Trudność:** ⭐⭐⭐

Wyciągnij ostatni stage z `rf_model.stages` (obiekt `RandomForestClassificationModel`) i pobierz z niego wektor `featureImportances`. Wypisz liczbę cech oraz 5 największych wartości ważności.

> 💡 **Hint:** `rf_model.stages[-1]` to wytrenowany model lasu losowego. Atrybut `.featureImportances` zwraca `SparseVector`.

In [ ]:
# ✏️ Ćwiczenie 4: Feature importances

# TODO: pobierz ostatni stage pipeline'u (wytrenowany RandomForestClassificationModel)
rf_classification_model = ...  # Zastąp ... swoim kodem

# TODO: pobierz wektor ważności cech
importances = ...  # Zastąp ... swoim kodem

print(f"Liczba cech w modelu: {importances.size}")
print(f"Top-5 wartości ważności: {sorted(importances.toArray(), reverse=True)[:5]}")

In [ ]:
# @title ✅ Test — Ćwiczenie 4

try:
    assert importances is not ..., "❌ importances nie zostało uzupełnione"
    assert importances.size > 0, "❌ Wektor ważności cech nie powinien być pusty"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 5: Accuracy, precision i recall

**Trudność:** ⭐⭐⭐

Użyj `MulticlassClassificationEvaluator`, aby obliczyć `accuracy`, `weightedPrecision` i `weightedRecall` dla `rf_predictions` z Ćwiczenia 2.

> 💡 **Hint:** `MulticlassClassificationEvaluator` działa również dla klasyfikacji binarnej — wystarczy ustawić `labelCol` i `predictionCol`.

In [ ]:
# ✏️ Ćwiczenie 5: Accuracy, precision, recall

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="clicked", predictionCol="prediction", metricName="accuracy"
)
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="clicked", predictionCol="prediction", metricName="weightedPrecision"
)
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="clicked", predictionCol="prediction", metricName="weightedRecall"
)

# TODO: oblicz metryki na rf_predictions używając powyższych evaluatorów
accuracy = ...  # Zastąp ... swoim kodem
precision = ...  # Zastąp ... swoim kodem
recall = ...  # Zastąp ... swoim kodem

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 5

try:
    for name, value in [("accuracy", accuracy), ("precision", precision), ("recall", recall)]:
        assert value is not ..., f"❌ {name} nie zostało uzupełnione"
        assert 0.0 <= value <= 1.0, f"❌ {name} powinno być w zakresie [0, 1]"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 6: Inspekcja DataFrame z predykcjami

**Trudność:** ⭐⭐⭐

Wybierz z `rf_predictions` kolumny `ad_position`, `device_type`, `clicked`, `prediction` i `probability`, i wyświetl 10 przykładowych wierszy jako `sample_predictions`.

> 💡 **Hint:** Użyj `.select(...)` i `.show(10, truncate=False)`.

In [ ]:
# ✏️ Ćwiczenie 6: Inspekcja predykcji

# TODO: wybierz kolumny ad_position, device_type, clicked, prediction, probability z rf_predictions
sample_predictions = ...  # Zastąp ... swoim kodem

sample_predictions.show(10, truncate=False)

In [ ]:
# @title ✅ Test — Ćwiczenie 6

try:
    expected_cols = {"ad_position", "device_type", "clicked", "prediction"}
    assert expected_cols.issubset(set(sample_predictions.columns)), "❌ Brakuje wymaganych kolumn w sample_predictions"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 7: Wpływ pozycji reklamy i urządzenia na CTR

**Trudność:** ⭐⭐⭐⭐

Pogrupuj `rf_predictions` po `ad_position` i `device_type`, i policz **średnią rzeczywistą** wartość `clicked` (`rzeczywisty_ctr`) oraz **średnią przewidywaną** wartość `prediction` (`przewidywany_ctr`). Posortuj malejąco po `rzeczywisty_ctr` jako `position_effect`.

**Oczekiwany wynik:** kombinacja `ad_position="top"` + `device_type="mobile"` powinna mieć wyższy `rzeczywisty_ctr` niż `ad_position="bottom"` + `device_type="desktop"`.

> 💡 **Hint:** Użyj `F.avg("clicked")` i `F.avg("prediction")` w `.agg(...)`.

In [ ]:
# ✏️ Ćwiczenie 7: Wpływ pozycji reklamy na CTR

from pyspark.sql import functions as F

# TODO: pogrupuj rf_predictions po ad_position i device_type, policz średni clicked i średnią prediction
position_effect = ...  # Zastąp ... swoim kodem

position_effect.show()

In [ ]:
# @title ✅ Test — Ćwiczenie 7

try:
    expected_cols = {"ad_position", "device_type", "rzeczywisty_ctr", "przewidywany_ctr"}
    assert expected_cols.issubset(set(position_effect.columns)), "❌ Brakuje wymaganych kolumn w position_effect"

    rows = {(r["ad_position"], r["device_type"]): r["rzeczywisty_ctr"] for r in position_effect.collect()}
    top_mobile = rows.get(("top", "mobile"))
    bottom_desktop = rows.get(("bottom", "desktop"))

    assert top_mobile is not None and bottom_desktop is not None, "❌ Brakuje oczekiwanych kombinacji w wyniku"
    assert top_mobile > bottom_desktop, "❌ 'top' + 'mobile' powinno mieć wyższy CTR niż 'bottom' + 'desktop'"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## 🏆 Bonus Challenge (opcjonalnie)

Zapisz wytrenowany `rf_model` (`PipelineModel`) na dysk, wczytaj go ponownie i zweryfikuj, że predykcje po wczytaniu są identyczne co do liczby wierszy z oryginalnymi `rf_predictions`.

> 💡 **Hint:** `PipelineModel` ma metody `.write().overwrite().save(path)` oraz klasową metodę `PipelineModel.load(path)`.

In [ ]:
# 🏆 Bonus: zapis i odczyt modelu

from pyspark.ml import PipelineModel

model_path = "/tmp/ctr_rf_pipeline_model"

# TODO: zapisz rf_model pod model_path (nadpisując, jeśli już istnieje)
...

# TODO: wczytaj model z powrotem jako loaded_model
loaded_model = ...  # Zastąp ... swoim kodem

reloaded_predictions = loaded_model.transform(test_df)

assert reloaded_predictions.count() == rf_predictions.count(), "❌ Liczba wierszy po wczytaniu modelu się nie zgadza"
print("✅ Model poprawnie zapisany i wczytany! 🎉")

---

## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Kodowanie zmiennych kategorycznych `StringIndexer` + `OneHotEncoder`
- ✅ Budowę pełnego `Pipeline` MLlib (preprocessing + model)
- ✅ Trening i porównanie `LogisticRegression` oraz `RandomForestClassifier`
- ✅ Ewaluację modeli: AUC, accuracy, precision, recall
- ✅ Analizę ważności cech i predykcji modelu
- ✅ Zapis i odczyt wytrenowanego `PipelineModel`

### 📚 Dalsze materiały
- [Spark MLlib Guide](https://spark.apache.org/docs/latest/ml-guide.html)
- [MLlib API Reference (pyspark.ml)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html)

### 🔮 Co dalej?
Na **Wykładzie 10 (08.12.2026)** poznamy zaawansowany MLlib — **tuning hiperparametrów** za pomocą `ParamGridBuilder` i `CrossValidator`!